# Avance Fase 3 — Semana 2

## Configuración inicial del proyecto

In [1]:
# =====================================================================
# CONFIGURACIÓN DEL PROYECTO
# =====================================================================

# Ruta escrita desde la raíz del proyecto, no desde la carpeta de este cuaderno
RUTA_DATOS = "data/processed/listings_procesado.csv"

COLUMNA_OBJETIVO = "price_clp"       # la variable que se quiere explicar
COLUMNA_ID = "id"                    # identificador: se elimina del análisis

COLUMNAS_CONTINUAS = [
    # ubicación
    "latitude", "longitude",
    # tamaño y capacidad
    "accommodates", "bedrooms", "beds", "banos_numero", "amenities_count",
    # reglas de estadía
    "minimum_nights", "maximum_nights",
    # disponibilidad y demanda
    "availability_30", "availability_60", "availability_90", "availability_365",
    "number_of_reviews", "number_of_reviews_ltm", "number_of_reviews_l30d",
    "reviews_per_month", "estimated_occupancy_l365d", "antiguedad_dias",
    # puntajes de reseñas
    "review_scores_rating", "review_scores_accuracy", "review_scores_cleanliness",
    "review_scores_checkin", "review_scores_communication",
    "review_scores_location", "review_scores_value",
    # anfitrión
    "calculated_host_listings_count",
]

COLUMNAS_NOMINALES = ["neighbourhood_cleansed", "property_type", "propiedad_agrupada",
                      "host_is_superhost"]   # superhost viene como texto t/f

COLUMNAS_BINARIAS = [
    "bedrooms_imputado", "beds_imputado", "ocupacion_tope", "es_multianfitrion",
    "amen_kitchen", "amen_wifi", "amen_hot_water", "amen_hair_dryer",
    "amen_cooking_basics", "amen_dishes_and_silverware", "amen_hangers",
    "amen_bed_linens", "amen_tv", "amen_microwave", "amen_self_check_in",
    "amen_refrigerator", "amen_iron", "amen_hot_water_kettle", "amen_wine_glasses",
    "amen_elevator", "amen_extra_pillows_and_blankets", "amen_long_term_stays_allowed",
    "amen_oven", "amen_toaster",
]

# Columnas del archivo que se dejan fuera a propósito (ver notas abajo)
COLUMNAS_EXCLUIDAS = ["host_id", "room_type", "banos_compartidos", "first_review",
                      "last_review", "precio_por_huesped", "estimated_revenue_l365d"]

SEMILLA = 42
PROPORCION_PRUEBA = 0.2

# Datos de demostración: solo para revisar el cuaderno sin el archivo real.
# Con True, los resultados NO son los de su proyecto. En la entrega debe ir False.
PERMITIR_DEMOSTRACION = False
# =====================================================================

COLUMNAS_ESPERADAS = ([COLUMNA_ID, COLUMNA_OBJETIVO] + COLUMNAS_CONTINUAS
                      + COLUMNAS_NOMINALES + COLUMNAS_BINARIAS)

from pathlib import Path


def localizar_raiz(inicio=None):
    """Devuelve la raíz del proyecto.

    Jupyter abre el kernel en la carpeta donde está guardado el cuaderno (F3), no en la
    carpeta desde la que se lanzó JupyterLab. Si la carpeta de trabajo es una carpeta
    de fase, la raíz es la carpeta superior.
    """
    inicio = Path.cwd() if inicio is None else Path(inicio)
    return inicio.parent if inicio.name in {"F1", "F2", "F3", "F4"} else inicio


RAIZ = localizar_raiz()
RUTA_DATOS = str(RAIZ / RUTA_DATOS)
DIR_PROCESADO = str(RAIZ / "data" / "processed")   # salidas compartidas por las fases

print("Raíz del proyecto   :", RAIZ.resolve())
print("Archivo configurado :", RUTA_DATOS)
print("Variable objetivo   :", COLUMNA_OBJETIVO)
print("Columnas esperadas  :", len(COLUMNAS_ESPERADAS))

Raíz del proyecto   : C:\Users\alfre\proyecto-grupo4-mcdi500
Archivo configurado : C:\Users\alfre\proyecto-grupo4-mcdi500\data\processed\listings_procesado.csv
Variable objetivo   : price_clp
Columnas esperadas  : 57


## Preparación del entorno

In [2]:
import os
import sys
import time
import timeit
import tracemalloc

import numpy as np
import pandas as pd

SEMILLA = 42
np.random.seed(SEMILLA)

print("Python :", sys.executable)
print("pandas :", pd.__version__)
print("NumPy  :", np.__version__)
print("Semilla:", SEMILLA)

Python : C:\Users\alfre\proyecto-grupo4-mcdi500\.venv\Scripts\python.exe
pandas : 3.1.1
NumPy  : 2.5.4
Semilla: 42


## Carga del data set

**Se realiza una carga de datos para verificar que se este leyendo correctamente desde la carpeta externa en la que se encuentra**

In [3]:
def leer_archivo(ruta):
    """Lee el archivo según su extensión y devuelve un DataFrame."""
    extension = os.path.splitext(ruta)[1].lower()

    if extension in (".csv", ".txt"):
        # sep=None con engine="python" infiere el separador: útil con datos
        # públicos chilenos, que suelen venir con punto y coma
        return pd.read_csv(ruta, sep=None, engine="python", encoding="utf-8")
    if extension in (".xlsx", ".xls"):
        return pd.read_excel(ruta)
    if extension == ".parquet":
        return pd.read_parquet(ruta)

    raise ValueError(
        f"Extensión no reconocida: '{extension}'. "
        "Se admiten .csv, .txt, .xlsx, .xls y .parquet."
    )


def verificar_esquema(df, columnas_esperadas):
    """Comprueba que estén todas las columnas declaradas en la configuración."""
    faltantes = [c for c in columnas_esperadas if c not in df.columns]
    if faltantes:
        raise KeyError(
            f"Faltan columnas declaradas en la configuración: {faltantes}\n"
            f"Columnas disponibles en el archivo: {list(df.columns)}"
        )
    sobrantes = [c for c in df.columns if c not in columnas_esperadas]
    return {"declaradas": len(columnas_esperadas),
            "en_archivo": df.shape[1],
            "no_declaradas": sobrantes}
def generar_demostracion(n=5110, semilla=SEMILLA):
    """Datos con el mismo esquema, solo para que el cuaderno corra sin el archivo.

    NO usar en el proyecto: sirve para revisar el cuaderno antes de tener
    los datos a mano.
    """
    rng = np.random.default_rng(semilla)
    df = pd.DataFrame({
        "id": rng.choice(np.arange(1, n * 20), size=n, replace=False),
        "gender": rng.choice(["Male", "Female", "Other"], n, p=[0.41, 0.585, 0.005]),
        "age": np.round(rng.uniform(0.08, 82, n), 1),
        "hypertension": rng.choice([0, 1], n, p=[0.90, 0.10]),
        "heart_disease": rng.choice([0, 1], n, p=[0.95, 0.05]),
        "ever_married": rng.choice(["Yes", "No"], n, p=[0.66, 0.34]),
        "work_type": rng.choice(
            ["Private", "Self-employed", "Govt_job", "children", "Never_worked"],
            n, p=[0.57, 0.16, 0.13, 0.13, 0.01]),
        "Residence_type": rng.choice(["Urban", "Rural"], n, p=[0.51, 0.49]),
        "avg_glucose_level": np.round(rng.uniform(55, 272, n), 2),
        "bmi": np.round(rng.normal(28.9, 7.9, n), 1),
        "smoking_status": rng.choice(
            ["never smoked", "Unknown", "formerly smoked", "smokes"],
            n, p=[0.37, 0.30, 0.17, 0.16]),
        "stroke": rng.choice([0, 1], n, p=[0.951, 0.049]),
    })
    df.loc[rng.choice(df.index, size=201, replace=False), "bmi"] = np.nan
    return df


def cargar(ruta=RUTA_DATOS, columnas_esperadas=None, permitir_demo=PERMITIR_DEMOSTRACION):
    """Carga el conjunto desde el archivo externo y verifica su esquema."""
    columnas_esperadas = columnas_esperadas or COLUMNAS_ESPERADAS

    if os.path.exists(ruta):
        df = leer_archivo(ruta)
        print(f"Archivo leído: {ruta}")
    elif permitir_demo:
        df = generar_demostracion()
        print("AVISO: no se encontró el archivo. Se usan DATOS DE DEMOSTRACIÓN.")
        print(f"       Coloquen su archivo en: {ruta}")
        print("       Y pongan PERMITIR_DEMOSTRACION = False en la configuración.")
    else:
        raise FileNotFoundError(
            f"No se encontró el archivo: {ruta}\n"
            f"Carpeta actual: {os.getcwd()}\n"
            "Revisen la variable RUTA_DATOS en la celda de configuración. "
            "La ruta se escribe desde la raíz del proyecto."
        )

    # Las columnas numéricas se fuerzan a número: un texto suelto las
    # convertiría en columna de objetos sin ningún aviso
    for columna in COLUMNAS_CONTINUAS:
        if columna in df.columns:
            df[columna] = pd.to_numeric(df[columna], errors="coerce")

    informe = verificar_esquema(df, columnas_esperadas)
    print(f"Forma: {df.shape[0]} filas x {df.shape[1]} columnas")
    print(f"Esquema verificado: {informe['declaradas']} columnas declaradas, "
          f"{len(informe['no_declaradas'])} no declaradas")
    if informe["no_declaradas"]:
        print("  No declaradas:", informe["no_declaradas"])
    return df


datos = cargar()
datos.head()

Archivo leído: C:\Users\alfre\proyecto-grupo4-mcdi500\data\processed\listings_procesado.csv
Forma: 3313 filas x 64 columnas
Esquema verificado: 57 columnas declaradas, 7 no declaradas
  No declaradas: ['host_id', 'room_type', 'banos_compartidos', 'precio_por_huesped', 'estimated_revenue_l365d', 'first_review', 'last_review']


,id,host_id,neighbourhood_cleansed,latitude,longitude,property_type,propiedad_agrupada,room_type,accommodates,bedrooms,...,review_scores_rating,review_scores_accuracy,review_scores_cleanliness,review_scores_checkin,review_scores_communication,review_scores_location,review_scores_value,host_is_superhost,calculated_host_listings_count,es_multianfitrion
0,1226526,828029c6c29f,Providencia,-33.42681,-70.60961,Entire rental unit,Entire rental unit,Entire home/apt,4,2.0,...,4.67,4.74,4.83,4.91,4.92,4.94,4.72,t,2,1
1,1300359,250fdbee5201,Providencia,-33.43005,-70.61772,Entire rental unit,Entire rental unit,Entire home/apt,4,2.0,...,4.73,4.71,4.80,4.79,4.85,4.85,4.61,f,20,1
2,3735992,481459247d1a,Estación Central,-33.45593,-70.70020,Entire condo,Entire condo,Entire home/apt,5,2.0,...,4.71,4.81,4.69,4.76,4.90,4.28,4.62,t,9,1
3,3810101,9e622c2063be,Lo Barnechea,-33.35603,-70.24830,Entire rental unit,Entire rental unit,Entire home/apt,5,2.0,...,4.91,4.89,4.71,4.89,4.86,4.89,4.77,f,1,0
4,1446357,476b08a4a2a1,Lo Barnechea,-33.34869,-70.29280,Entire condo,Entire condo,Entire home/apt,6,2.0,...,4.68,4.64,4.51,4.67,4.89,4.83,4.42,f,1,0


**Se perfilan los datos del data set**

In [6]:
def perfilar(df):
    """Devuelve una fila por columna con su tipo, únicos y porcentaje de nulos."""
    return pd.DataFrame({
        "columna": df.columns,
        "tipo": [str(t) for t in df.dtypes],
        "unicos": [df[c].nunique(dropna=True) for c in df.columns],
        "nulos": df.isna().sum().values,
        "pct_nulos": (df.isna().mean() * 100).round(2).values,
    }).sort_values("pct_nulos", ascending=False).reset_index(drop=True)


perfil = perfilar(datos)
faltantes = perfil[perfil["nulos"] > 0]
print("Columnas con valores faltantes:")
if faltantes.empty:
    print("  Ninguna: el conjunto de datos no tiene valores faltantes.")
else:
    print(faltantes.to_string(index=False))

# La variable objetivo es continua: se describe con estadísticos, no con frecuencias
print(f"\nDistribución de la variable objetivo '{COLUMNA_OBJETIVO}':")
print(datos[COLUMNA_OBJETIVO].describe().round(2))
print(f"Asimetría (skew)          : {datos[COLUMNA_OBJETIVO].skew():.2f}")
print(f"Asimetría tras log1p      : {np.log1p(datos[COLUMNA_OBJETIVO]).skew():.2f}")

print("\nCategorías de property_type (ordenadas por frecuencia):")
print(datos["property_type"].value_counts())

print("\nCategorías de propiedad_agrupada:")
print(datos["propiedad_agrupada"].value_counts())

Columnas con valores faltantes:
  Ninguna: el conjunto de datos no tiene valores faltantes.

Distribución de la variable objetivo 'price_clp':
count        3313.00
mean        98274.27
std        650820.17
min         19125.00
25%         44760.00
50%         59950.00
75%         85500.00
max      37034021.50
Name: price_clp, dtype: float64
Asimetría (skew)          : 55.25
Asimetría tras log1p      : 1.83

Categorías de property_type (ordenadas por frecuencia):
property_type
Entire rental unit           2926
Entire condo                  168
Entire serviced apartment      76
Entire home                    66
Entire loft                    30
Entire vacation home            9
Entire cabin                    8
Entire guesthouse               8
Tiny home                       7
Entire guest suite              4
Entire chalet                   3
Entire villa                    2
Camper/RV                       1
Entire cottage                  1
Casa particular                 1
Windmill 